# TP — Module 3 : ML Classique Spatial
## Prédire le niveau d'accès à l'eau potable — Burkina Faso

> **Cours** : ASML — Version 1 | **Institut** : 2iE

---

## Différence avec le CM

| | CM | TP |
|-|----|----|
| **Variable cible** | ipc_phase (insécurité alimentaire 1–5) | acces_eau (0=Faible / 1=Moyen / 2=Bon) |
| **Question** | Prédire les phases IPC | Prédire le niveau d'accès à l'eau |
| **Contexte** | CILSS / Cadre Harmonisé | DGRE / Programme WASH |
| **Défi spécifique** | 5 classes déséquilibrées | 3 classes + variable ordinal |

Les **outils sont identiques** (RF, XGBoost, Spatial CV, Kappa).
Le contexte, les données et les questions sont différents.

---

## Contexte de la mission

La DGRE (Direction Générale des Ressources en Eau) du BF souhaite prédire
le **niveau d'accès à l'eau potable** dans les provinces non couvertes par
les enquêtes JMP récentes. La variable cible est :

- `0` = Faible accès (taux < 45%)
- `1` = Accès moyen (45–70%)
- `2` = Bon accès (> 70%)

Les features disponibles sont des variables géospatiales mesurables
sans enquête terrain : densité de points d'eau, enclavement, pluviométrie.

---

## Données fournies

| Feature | Description |
|---------|-------------|
| `puits_par_10000hab` | Densité de points d'eau / 10 000 habitants |
| `pop_density_tp` | Densité de population (hab/km²) |
| `compacite` | Compacité de Polsby-Popper |
| `dist_route_km` | Distance au réseau routier (km) |
| `travel_time_h` | Temps de trajet vers le centre de santé (h) |
| `pluvio_mm` | Pluviométrie annuelle moyenne (mm/an) |

In [ ]:
# !pip install scikit-learn xgboost imbalanced-learn --quiet

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, cohen_kappa_score, f1_score)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance

TP3_JSON = '[{"province":"Oudalan","region":"Sahel","nb_puits":45,"pop":322000,"superficie_km2":22236,"compacite":0.42,"dist_route_km":245,"travel_time_h":4.2,"pluvio_mm":280,"lat":14.48,"acces_eau":0,"pop_density_tp":14.4810217665,"puits_par_10000hab":1.397515528},{"province":"Séno","region":"Sahel","nb_puits":62,"pop":325000,"superficie_km2":13484,"compacite":0.55,"dist_route_km":198,"travel_time_h":3.8,"pluvio_mm":310,"lat":14.1,"acces_eau":0,"pop_density_tp":24.1026401661,"puits_par_10000hab":1.9076923077},{"province":"Soum","region":"Sahel","nb_puits":78,"pop":369000,"superficie_km2":15704,"compacite":0.48,"dist_route_km":220,"travel_time_h":4.0,"pluvio_mm":350,"lat":13.98,"acces_eau":0,"pop_density_tp":23.4971981661,"puits_par_10000hab":2.1138211382},{"province":"Yagha","region":"Sahel","nb_puits":38,"pop":214000,"superficie_km2":12376,"compacite":0.62,"dist_route_km":185,"travel_time_h":3.5,"pluvio_mm":420,"lat":13.35,"acces_eau":0,"pop_density_tp":17.2915319974,"puits_par_10000hab":1.7757009346},{"province":"Loroum","region":"Nord","nb_puits":55,"pop":194000,"superficie_km2":5549,"compacite":0.71,"dist_route_km":120,"travel_time_h":2.8,"pluvio_mm":480,"lat":13.7,"acces_eau":1,"pop_density_tp":34.9612542801,"puits_par_10000hab":2.8350515464},{"province":"Yatenga","region":"Nord","nb_puits":95,"pop":535000,"superficie_km2":7275,"compacite":0.68,"dist_route_km":85,"travel_time_h":2.2,"pluvio_mm":520,"lat":13.55,"acces_eau":1,"pop_density_tp":73.5395189003,"puits_par_10000hab":1.7757009346},{"province":"Passoré","region":"Nord","nb_puits":82,"pop":313000,"superficie_km2":3957,"compacite":0.74,"dist_route_km":65,"travel_time_h":1.9,"pluvio_mm":560,"lat":12.89,"acces_eau":1,"pop_density_tp":79.1003285317,"puits_par_10000hab":2.6198083067},{"province":"Sanmatenga","region":"Centre-Nord","nb_puits":88,"pop":532000,"superficie_km2":8860,"compacite":0.65,"dist_route_km":72,"travel_time_h":1.8,"pluvio_mm":590,"lat":13.05,"acces_eau":1,"pop_density_tp":60.0451467269,"puits_par_10000hab":1.6541353383},{"province":"Namentenga","region":"Centre-Nord","nb_puits":72,"pop":331000,"superficie_km2":6670,"compacite":0.58,"dist_route_km":95,"travel_time_h":2.1,"pluvio_mm":620,"lat":13.1,"acces_eau":1,"pop_density_tp":49.6251874063,"puits_par_10000hab":2.1752265861},{"province":"Bam","region":"Centre-Nord","nb_puits":65,"pop":295000,"superficie_km2":5869,"compacite":0.72,"dist_route_km":80,"travel_time_h":1.7,"pluvio_mm":640,"lat":13.45,"acces_eau":1,"pop_density_tp":50.2640995059,"puits_par_10000hab":2.2033898305},{"province":"Kadiogo","region":"Centre","nb_puits":210,"pop":2415000,"superficie_km2":2585,"compacite":0.81,"dist_route_km":12,"travel_time_h":0.4,"pluvio_mm":820,"lat":12.36,"acces_eau":2,"pop_density_tp":934.2359767892,"puits_par_10000hab":0.8695652174},{"province":"Bazega","region":"Centre-Sud","nb_puits":58,"pop":226000,"superficie_km2":3706,"compacite":0.76,"dist_route_km":45,"travel_time_h":1.1,"pluvio_mm":850,"lat":12.1,"acces_eau":2,"pop_density_tp":60.9821910416,"puits_par_10000hab":2.5663716814},{"province":"Zoundweogo","region":"Centre-Sud","nb_puits":62,"pop":244000,"superficie_km2":2590,"compacite":0.79,"dist_route_km":55,"travel_time_h":1.3,"pluvio_mm":860,"lat":11.85,"acces_eau":2,"pop_density_tp":94.2084942085,"puits_par_10000hab":2.5409836066},{"province":"Nahouri","region":"Centre-Sud","nb_puits":55,"pop":185000,"superficie_km2":3918,"compacite":0.66,"dist_route_km":70,"travel_time_h":1.6,"pluvio_mm":870,"lat":11.5,"acces_eau":2,"pop_density_tp":47.2179683512,"puits_par_10000hab":2.972972973},{"province":"Boulgou","region":"Centre-Est","nb_puits":75,"pop":398000,"superficie_km2":5868,"compacite":0.59,"dist_route_km":110,"travel_time_h":2.3,"pluvio_mm":780,"lat":11.85,"acces_eau":1,"pop_density_tp":67.8254942059,"puits_par_10000hab":1.8844221106},{"province":"Ganzourgou","region":"Plateau C.","nb_puits":68,"pop":255000,"superficie_km2":5226,"compacite":0.67,"dist_route_km":58,"travel_time_h":1.4,"pluvio_mm":750,"lat":12.35,"acces_eau":2,"pop_density_tp":48.794489093,"puits_par_10000hab":2.6666666667},{"province":"Houet","region":"Hauts-B.","nb_puits":145,"pop":1015000,"superficie_km2":10200,"compacite":0.61,"dist_route_km":22,"travel_time_h":0.7,"pluvio_mm":980,"lat":11.18,"acces_eau":2,"pop_density_tp":99.5098039216,"puits_par_10000hab":1.4285714286},{"province":"Comoé","region":"Cascades","nb_puits":82,"pop":274000,"superficie_km2":16000,"compacite":0.55,"dist_route_km":85,"travel_time_h":1.8,"pluvio_mm":1050,"lat":10.6,"acces_eau":2,"pop_density_tp":17.125,"puits_par_10000hab":2.9927007299},{"province":"Poni","region":"Sud-Ouest","nb_puits":78,"pop":267000,"superficie_km2":10075,"compacite":0.52,"dist_route_km":95,"travel_time_h":2.0,"pluvio_mm":1100,"lat":10.3,"acces_eau":2,"pop_density_tp":26.5012406948,"puits_par_10000hab":2.9213483146},{"province":"Noumbiel","region":"Sud-Ouest","nb_puits":45,"pop":100000,"superficie_km2":3562,"compacite":0.63,"dist_route_km":120,"travel_time_h":2.4,"pluvio_mm":1150,"lat":10.0,"acces_eau":1,"pop_density_tp":28.0741156654,"puits_par_10000hab":4.5}]'

df_tp = pd.read_json(TP3_JSON)

FEATURES_TP = ['puits_par_10000hab','pop_density_tp','compacite',
               'dist_route_km','travel_time_h','pluvio_mm']
TARGET_TP   = 'acces_eau'

print('Feature matrix accès eau :', df_tp.shape)
print('Distribution acces_eau :', dict(df_tp[TARGET_TP].value_counts().sort_index()))
print('Colonnes :', list(df_tp.columns))

---
## Exercice 1 — Baseline Random Forest (3 pts)

1. Créez `X_tp` et `y_tp` depuis la feature matrix DGRE
2. Entraînez un `RandomForestClassifier` avec `class_weight='balanced'`
3. Calculez sur les données d'entraînement : Kappa, F1-macro, rapport de classification
4. Tracez la **matrice de confusion**
5. Commentez : quelles classes sont les mieux / moins bien prédites ? Pourquoi ?

> 💡 Rappel : ces scores sont **biaisés** (entraînement = test).
> L'exercice 3 donnera les scores honnêtes via spatial CV.

In [ ]:
# ── Ex.1 : Votre code ici ──

# 1. Préparer X et y


# 2. Entraîner le Random Forest


# 3. Kappa + F1 + rapport


# 4. Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
# ...
plt.show()

# 5. Commentaire (en commentaire Python) :
# ...

---
## Exercice 2 — XGBoost et comparaison (2 pts)

1. Entraînez un `XGBClassifier` avec régularisation forte
   (`max_depth=3`, `learning_rate=0.05`, `n_estimators=100`)
2. Comparez les Kappa RF vs XGBoost **sur les données d'entraînement**
3. Comparez les **feature importances Gini** des deux modèles
   - Tracez un graphe à barres horizontales pour chacun
   - La pluviométrie (`pluvio_mm`) est-elle plus importante que
     la densité de puits (`puits_par_10000hab`) ?

> ⚠️ Rappel : XGBoost nécessite `LabelEncoder` (classes 0..n-1).
> Ici `acces_eau` est déjà 0/1/2 — vérifier avant d'encoder.

In [ ]:
# ── Ex.2 : Votre code ici ──

# 1. XGBoost


# 2. Comparaison Kappa


# 3. Comparaison feature importances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# ...
plt.show()

# Réponse pluvio vs puits :
# ...

---
## Exercice 3 — Spatial Cross-Validation (3 pts)

1. Définissez 4 blocs géographiques pour les 20 provinces TP
   en vous basant sur les régions (`region`) ou la latitude (`lat`)
2. Implémentez la **spatial block CV** (fonction `spatial_block_cv`
   vue en CM — ou votre propre version)
3. Comparez : Kappa k-fold classique vs Kappa spatial CV
   pour les deux modèles (RF et XGBoost)
4. Quel modèle recommandez-vous à la DGRE ? Justifiez avec les métriques.

> 📊 **Question analytique** : si le Kappa spatial CV est très inférieur au
> Kappa k-fold, que cela indique-t-il sur la structure spatiale de l'accès
> à l'eau au Burkina Faso ? Quel lien avec le Moran I ?

In [ ]:
# ── Ex.3 : Votre code ici ──

# 1. Définition des blocs
BLOCS_TP = {
    # ...
}

# 2. Spatial Block CV


# 3. Comparaison k-fold vs spatial CV


# 4. Recommandation (commentaire) :
# ...

# Réponse analytique :
# ...

---
## Exercice 4 — Carte prédictive et rapport DGRE (2 pts)

1. Entraînez le **meilleur modèle** (choisi à l'exercice 3) sur toutes les données
2. Ajoutez les prédictions au DataFrame : `acces_pred`
3. Identifiez les **3 provinces les plus mal prédites** (écart |prédit - observé| ≥ 1)
4. Produisez un **graphe de synthèse** : accès observé vs prédit pour chaque province
   (scatter plot ou barres côte à côte)
5. Rédigez en commentaire une **recommandation de 3 lignes** pour la DGRE :
   - Quel modèle utiliser ?
   - Dans quelles provinces les prédictions sont-elles fiables ?
   - Quelles limites doit connaître un décideur ?

In [ ]:
# ── Ex.4 : Votre code ici ──

# 1. Entraînement final du meilleur modèle


# 2. Prédictions


# 3. Provinces mal prédites


# 4. Graphe de synthèse
fig, ax = plt.subplots(figsize=(10, 5))
# ...
plt.show()

# 5. Recommandation DGRE :
# - Modèle recommandé : ...
# - Provinces fiables : ...
# - Limites : ...

---
## Conclusion du TP

| Exercice | Compétence évaluée |
|----------|-------------------|
| 1 | RF + class_weight + diagnostics (Kappa, F1-macro, confusion matrix) |
| 2 | XGBoost + comparaison feature importance |
| 3 | Spatial Cross-Validation + interprétation biais k-fold |
| 4 | Carte prédictive + rapport décisionnel |